# Stage 5. Multi-Task Training and Model Comparison

Notebook ini melatih model multi-task tiga head (regresi hemoglobin, klasifikasi anemia, severity ordinal) pada fitur nail memakai k-fold cross-validation berbasis pasien, membandingkan konfigurasi Path A saja, Path B saja, dan Full Fusion memakai src.common.train.run_kfold yang situs-agnostik. Head klasifikasi anemia biner diperlakukan sebagai output utama pada situs ini, sedangkan severity granular WHO 2024 tetap dilatih namun dilaporkan dengan disclaimer eksperimen sekunder, mengikuti keputusan arsitektur di RENCANA_PIPELINE_KUKU.md yang didasari literatur bahwa sinyal warna nail bed jauh lebih noisy dibanding konjungtiva untuk gradasi halus.

## Environment Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import numpy as np
import pandas as pd
import torch

from configs import paths
from src.common import manifest as manifest_utils, train

output_dir = paths.outputs_dir("nail")
artifact_dir = paths.artifacts_dir("nail")
manifest = pd.read_csv(output_dir / "manifest.csv")
manifest = manifest[manifest["roi_precropped"]].reset_index(drop=True)
manifest = manifest_utils.assign_kfold(manifest, n_splits=5, seed=42)

handcrafted = pd.read_csv(output_dir / "handcrafted_features.csv")
deep_embeddings = np.load(output_dir / "deep_embeddings.npy")
embedding_uids = pd.read_csv(output_dir / "deep_embeddings_uids.csv")["uid"].tolist()
print("sampel training", len(manifest))
print("polish_flag True", int(manifest["polish_flag"].sum()), "dari", len(manifest))

## Sanity Check: Handcrafted Features with Classical SVM

SVM RBF pada fitur hand-crafted terstandardisasi sebagai baseline klasik sebelum model deep, memberi acuan performa minimum yang harus dilampaui oleh model multi-task fusi.

In [2]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

svm_features = StandardScaler().fit_transform(handcrafted.drop(columns=["uid"]))
svm_labels = manifest.set_index("uid").loc[handcrafted["uid"], "anemic"].to_numpy()
svm_scores = cross_val_score(SVC(kernel="rbf"), svm_features, svm_labels, cv=5, scoring="accuracy")
print("SVM accuracy per fold", np.round(svm_scores, 3))
print("SVM mean accuracy", round(svm_scores.mean(), 3))

SVM accuracy per fold [0.646 0.652 0.652 0.675 0.681]
SVM mean accuracy 0.661


## Model Configurations

Tujuh konfigurasi diuji dengan protokol pelatihan identik lewat run_kfold, disusun sebagai eksperimen berjenjang dengan hipotesis eksplisit, bukan pencarian acak. Empat konfigurasi pertama (path_a_handcrafted, path_b_deep, full_fusion, full_fusion_polish) adalah baseline jalur fitur seperti sebelumnya. Tiga konfigurasi tambahan menguji hipotesis spesifik penyebab performa mentok di sekitar 64 persen akurasi, sebelum accuracy diketahui menyesatkan (lihat markdown AUC di bawah):

Hipotesis 1, classification_emphasis, menaikkan bobot loss klasifikasi anemia relatif terhadap regresi dan severity (loss_weights 0.5, 2.0, 0.3 dibanding default 1.0, 1.0, 0.5), menguji apakah head klasifikasi kalah rebutan gradien terhadap dua head lain pada training multi-task bersama.

Hipotesis 2, no_severity_head, mematikan total kontribusi loss severity (loss_weights 1.0, 1.0, 0.0), menguji apakah head severity (kelas Moderate cuma 8.8 persen, sinyal noisy) justru mengganggu representasi bersama yang dipakai head klasifikasi anemia.

Hipotesis 3, full_fusion_small_trunk, mengecilkan kapasitas trunk (trunk_dim 64, dropout 0.4 dibanding default 128 dan 0.3), menguji apakah model overfit karena dimensi fitur fusi penuh (286) relatif besar terhadap jumlah sampel train per fold (sekitar 640).

In [ ]:
configurations = {
    "path_a_handcrafted": dict(use_handcrafted=True, use_deep=False),
    "path_b_deep": dict(use_handcrafted=False, use_deep=True),
    "full_fusion": dict(use_handcrafted=True, use_deep=True),
    "full_fusion_polish": dict(use_handcrafted=True, use_deep=True, extra_columns=["polish_flag"]),
    "classification_emphasis": dict(use_handcrafted=True, use_deep=True, loss_weights=(0.5, 2.0, 0.3)),
    "no_severity_head": dict(use_handcrafted=True, use_deep=True, loss_weights=(1.0, 1.0, 0.0)),
    "full_fusion_small_trunk": dict(use_handcrafted=True, use_deep=True, trunk_dim=64, dropout=0.4),
}

## Train and Evaluate All Configurations

In [ ]:
results = {}
for name, overrides in configurations.items():
    results[name] = train.run_kfold(
        manifest, handcrafted, deep_embeddings, embedding_uids,
        n_splits=5, epochs=60, **overrides,
    )
    fold_metrics = results[name]["fold_metrics"]
    print(
        name,
        "MAE", round(fold_metrics["mae"].mean(), 3),
        "accuracy", round(fold_metrics["accuracy"].mean(), 3),
        "precision", round(fold_metrics["precision"].mean(), 3),
        "recall", round(fold_metrics["recall"].mean(), 3),
        "f1", round(fold_metrics["f1"].mean(), 3),
        "auc", round(fold_metrics["auc"].mean(), 3),
        "balanced_accuracy", round(fold_metrics["balanced_accuracy"].mean(), 3),
    )

## Diagnosis: Accuracy Menyesatkan pada Kelas Timpang

Prevalensi anemic pada dataset nail hanya 32.6 persen (262 dari 803), sehingga baseline trivial (selalu memprediksi non-anemic) sudah mencapai accuracy 67.4 persen, lebih tinggi dari accuracy model multi-task manapun di atas. Ini bukan berarti model tidak belajar apa pun, akurasi memang metrik yang tidak adil dipakai sendirian pada kelas timpang. AUC (tidak bergantung ambang keputusan maupun prevalensi) dan balanced accuracy dipakai sebagai pembanding yang jujur, dibandingkan pula terhadap baseline klasik (logistic regression dan SVM RBF) pada fitur yang sama untuk memastikan sinyal dari model deep memang lebih baik daripada model klasik, bukan sekadar berbeda ambang keputusan.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
deep_lookup = {uid: row for uid, row in zip(embedding_uids, deep_embeddings)}
X_deep = np.stack([deep_lookup[uid] for uid in handcrafted["uid"]])

classical_baselines = {
    "handcrafted_logreg": (StandardScaler().fit_transform(handcrafted.drop(columns=["uid"])), LogisticRegression(max_iter=2000, class_weight="balanced")),
    "deep_logreg": (StandardScaler().fit_transform(X_deep), LogisticRegression(max_iter=2000, class_weight="balanced")),
    "handcrafted_svm_rbf": (StandardScaler().fit_transform(handcrafted.drop(columns=["uid"])), SVC(kernel="rbf", probability=True)),
}
print("baseline trivial (selalu non-anemic): accuracy", round((svm_labels == 0).mean(), 3))
for name, (X, clf) in classical_baselines.items():
    proba = cross_val_predict(clf, X, svm_labels, cv=skf, method="predict_proba")[:, 1]
    print(name, "AUC", round(roc_auc_score(svm_labels, proba), 3))
for name, result in results.items():
    fold_metrics = result["fold_metrics"]
    print(name, "AUC", round(fold_metrics["auc"].mean(), 3), "balanced_accuracy", round(fold_metrics["balanced_accuracy"].mean(), 3))

## Severity Head: Secondary Experiment Disclaimer

Kelas Moderate hanya 71 dari 803 sampel (8.8 persen), minoritas keras yang diprediksi literatur (Asare dkk. 2023, Navarro-Cabrera dkk. 2025) akan sulit dibedakan lewat sinyal warna nail bed saja. Akurasi severity dilaporkan di sini untuk kelengkapan, namun tidak diklaim setara reliabilitas dengan situs konjungtiva atau palm, sesuai keputusan arsitektur di RENCANA_PIPELINE_KUKU.md. Cohen's kappa dilaporkan bersamaan dengan akurasi karena akurasi saja tidak sensitif terhadap kegagalan model pada kelas minoritas.

In [5]:
from sklearn.metrics import cohen_kappa_score


def _severity_kappa(oof):
    severity_valid = oof["severity_true"] >= 0
    if not severity_valid.any():
        return float("nan")
    return cohen_kappa_score(oof.loc[severity_valid, "severity_true"], oof.loc[severity_valid, "severity_pred"])


for name, result in results.items():
    fold_metrics = result["fold_metrics"]
    kappa = _severity_kappa(result["oof"])
    print(name, "severity accuracy", round(fold_metrics["severity_accuracy"].mean(), 3), "severity kappa", round(kappa, 3))

path_a_handcrafted severity accuracy 0.623 severity kappa 0.103
path_b_deep severity accuracy 0.674 severity kappa 0.002
full_fusion severity accuracy 0.616 severity kappa 0.144


## Compare Against Literature Baselines

Akurasi klasifikasi anemia dibandingkan terhadap studi analog yang memakai modalitas fingernail, karena tidak ada paper yang memodelkan populasi Valles-Coral/Navarro-Cabrera secara langsung untuk klasifikasi biner. Perbandingan ini punya keterbatasan penting, angka accuracy Peksi dkk. dan Asare dkk. tidak diketahui dihitung pada distribusi kelas seimbang atau tidak, sedangkan dataset nail di sini timpang (32.6 persen anemic), sehingga AUC dan F1 dilaporkan berdampingan sebagai pembanding yang lebih adil terhadap ketimpangan kelas.

In [ ]:
literature_baselines = {
    "Peksi et al. 2021 (Naive Bayes, nail+palm)": (0.875, 0.923),
    "Asare et al. 2023 (CNN, fingernail+palm+conjunctiva, Ghana)": (0.71, 0.98),
}
for name, result in results.items():
    fold_metrics = result["fold_metrics"]
    accuracy = fold_metrics["accuracy"].mean()
    f1 = fold_metrics["f1"].mean()
    auc = fold_metrics["auc"].mean()
    print(f"{name}: accuracy {accuracy:.3f}, f1 {f1:.3f}, auc {auc:.3f}")
for label, (low, high) in literature_baselines.items():
    print(f"{label}: {low:.3f} - {high:.3f}")

## Save Results

In [ ]:
import json

comparison_rows = []
for name, result in results.items():
    fold_metrics = result["fold_metrics"]
    comparison_rows.append({
        "configuration": name,
        "mae": fold_metrics["mae"].mean(),
        "rmse": fold_metrics["rmse"].mean(),
        "accuracy": fold_metrics["accuracy"].mean(),
        "precision": fold_metrics["precision"].mean(),
        "recall": fold_metrics["recall"].mean(),
        "f1": fold_metrics["f1"].mean(),
        "auc": fold_metrics["auc"].mean(),
        "balanced_accuracy": fold_metrics["balanced_accuracy"].mean(),
        "severity_accuracy": fold_metrics["severity_accuracy"].mean(),
        "severity_kappa": _severity_kappa(result["oof"]),
    })
comparison_table = pd.DataFrame(comparison_rows)
comparison_table.to_csv(output_dir / "multitask_model_comparison.csv", index=False)

for name, result in results.items():
    for fold_index, fold_model in enumerate(result["models"]):
        checkpoint_path = artifact_dir / f"multitask_{name}_fold{fold_index}.pt"
        torch.save(fold_model.state_dict(), checkpoint_path)

best_auc_name = comparison_table.sort_values("auc", ascending=False).iloc[0]["configuration"]
best_f1_name = comparison_table.sort_values("f1", ascending=False).iloc[0]["configuration"]
print("konfigurasi AUC tertinggi", best_auc_name)
print("konfigurasi F1 tertinggi (dipakai sebagai model resmi karena AUC tidak mencerminkan performa pada ambang default)", best_f1_name)
results[best_auc_name]["oof"].to_csv(output_dir / "multitask_oof_best_auc.csv", index=False)
results[best_f1_name]["oof"].to_csv(output_dir / "multitask_oof_best_f1.csv", index=False)
results["full_fusion"]["oof"].to_csv(output_dir / "multitask_oof_full_fusion.csv", index=False)

with open(output_dir / "multitask_official_best.json", "w") as handle:
    json.dump({"configuration": best_f1_name, "selection_criterion": "f1"}, handle, indent=2)

comparison_table.round(4)

## Hyperparameter Search with Optuna

Tujuh konfigurasi manual di atas semuanya mentok pada AUC 0.70 sampai 0.72 tidak peduli bobot loss atau kapasitas trunk diubah bagaimana pun, begitu pula fine-tuning end-to-end pada notebook 05b. Sebelum menerima ini sebagai batas sinyal nail bed, satu kemungkinan belum diuji, kombinasi hyperparameter yang belum dicoba manual (learning rate, jumlah epoch, kombinasi bobot loss dan kapasitas trunk secara bersamaan, bukan satu per satu). Pencarian ini memakai pendekatan yang sama dengan konjungtiva yang berhasil menaikkan akurasi dari 64 ke 73 persen lewat Optuna, diuji langsung pada data nail sendiri, bukan mewarisi asumsi bahwa nail akan merespons sama.

Objective yang diminimalkan adalah 1 dikurangi AUC rata-rata, bukan MAE atau accuracy, sesuai diagnosis bahwa AUC adalah metrik yang tidak menyesatkan pada dataset anemic yang timpang ini. Tidak ada komponen worst-case antar dataset seperti pada konjungtiva karena nail hanya berasal dari satu populasi (Valles-Coral), bukan gabungan beberapa dataset.

In [ ]:
import optuna


def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    epochs = trial.suggest_int("epochs", 30, 120)
    weight_classification = trial.suggest_float("weight_classification", 0.5, 3.0)
    weight_severity = trial.suggest_float("weight_severity", 0.1, 1.0)
    focal_gamma = trial.suggest_float("focal_gamma", 0.5, 3.0)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    trunk_dim = trial.suggest_categorical("trunk_dim", [64, 128, 256])
    attention_dim = trial.suggest_categorical("attention_dim", [32, 64, 128])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    result = train.run_kfold(
        manifest, handcrafted, deep_embeddings, embedding_uids,
        n_splits=5, epochs=epochs, batch_size=batch_size, learning_rate=learning_rate,
        loss_weights=(1.0, weight_classification, weight_severity),
        trunk_dim=trunk_dim, attention_dim=attention_dim, dropout=dropout, focal_gamma=focal_gamma,
        use_handcrafted=True, use_deep=True,
    )
    fold_metrics = result["fold_metrics"]
    auc_mean = float(fold_metrics["auc"].mean())
    balanced_accuracy_mean = float(fold_metrics["balanced_accuracy"].mean())
    score = 1.0 - auc_mean

    trial.set_user_attr("auc_mean", auc_mean)
    trial.set_user_attr("balanced_accuracy_mean", balanced_accuracy_mean)
    return score


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("best value", round(study.best_value, 4))
print("best params", study.best_params)
print("best auc_mean", round(study.best_trial.user_attrs["auc_mean"], 4))
print("best balanced_accuracy_mean", round(study.best_trial.user_attrs["balanced_accuracy_mean"], 4))

## Retrain with Best Hyperparameters

Full fusion dilatih ulang memakai hyperparameter terbaik hasil pencarian Optuna, lalu dibandingkan jujur dengan tujuh konfigurasi manual sebelumnya berdasarkan AUC, bukan MAE atau accuracy.

In [ ]:
import json

best_params = study.best_params
tuned_result = train.run_kfold(
    manifest, handcrafted, deep_embeddings, embedding_uids,
    n_splits=5,
    epochs=best_params["epochs"],
    batch_size=best_params["batch_size"],
    learning_rate=best_params["learning_rate"],
    loss_weights=(1.0, best_params["weight_classification"], best_params["weight_severity"]),
    trunk_dim=best_params["trunk_dim"],
    attention_dim=best_params["attention_dim"],
    dropout=best_params["dropout"],
    focal_gamma=best_params["focal_gamma"],
    use_handcrafted=True, use_deep=True,
)
tuned_metrics = tuned_result["fold_metrics"]
tuned_name = "full_fusion_tuned"

comparison_rows.append({
    "configuration": tuned_name,
    "mae": tuned_metrics["mae"].mean(),
    "rmse": tuned_metrics["rmse"].mean(),
    "accuracy": tuned_metrics["accuracy"].mean(),
    "precision": tuned_metrics["precision"].mean(),
    "recall": tuned_metrics["recall"].mean(),
    "f1": tuned_metrics["f1"].mean(),
    "auc": tuned_metrics["auc"].mean(),
    "balanced_accuracy": tuned_metrics["balanced_accuracy"].mean(),
    "severity_accuracy": tuned_metrics["severity_accuracy"].mean(),
    "severity_kappa": _severity_kappa(tuned_result["oof"]),
})
comparison_table = pd.DataFrame(comparison_rows)
comparison_table.to_csv(output_dir / "multitask_model_comparison.csv", index=False)

with open(output_dir / "multitask_optuna_best_params.json", "w") as handle:
    json.dump(
        {
            "best_params": best_params,
            "best_value": study.best_value,
            "auc_mean": study.best_trial.user_attrs["auc_mean"],
            "balanced_accuracy_mean": study.best_trial.user_attrs["balanced_accuracy_mean"],
        },
        handle, indent=2,
    )

previous_best_auc = comparison_table.loc[comparison_table["configuration"] != tuned_name, "auc"].max()
if tuned_metrics["auc"].mean() > previous_best_auc:
    tuned_result["oof"].to_csv(output_dir / "multitask_oof_best_auc.csv", index=False)
    for fold_index, fold_model in enumerate(tuned_result["models"]):
        checkpoint_path = artifact_dir / f"multitask_full_fusion_tuned_fold{fold_index}.pt"
        torch.save(fold_model.state_dict(), checkpoint_path)
    print("tuned checkpoints disimpan, AUC membaik dibanding konfigurasi manual terbaik", round(previous_best_auc, 4))
else:
    print("tuned checkpoints tidak menimpa, AUC manual terbaik masih lebih baik atau setara", round(previous_best_auc, 4))

comparison_table.round(4)